# Lecture 34 Worksheet: The Decoder, Masks and Training a Transformer
**NAVTTC / NIAI AI/ML Trainer Cohort** — runs in Google Colab or local Jupyter (PyTorch, CPU is enough, about 20 minutes in total). Fill every `None` / `# YOUR CODE` area; check cells print `OK` or tell you what to fix.

Today we finish the Transformer: the **decoder** (masked self-attention + cross-attention), **teacher forcing**, **greedy decoding**, and we train a small Transformer on the SAME Urdu→English corpus we used in Lectures 30 and 31, so we can compare it fairly with the LSTM encoder–decoder and the attention-RNN. The multi-head attention, feed-forward layer and encoder block from Lecture 33 are provided.

In [ ]:
import os, math, json, time, random
os.environ["OMP_NUM_THREADS"] = "2"
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
torch.set_num_threads(2); np.set_printoptions(precision=3, suppress=True)
torch.manual_seed(0); np.random.seed(0); random.seed(0)
R = {}          # results we collect, printed as one JSON line at the very end
def check(name, ok, hint=""):
    print(("OK   " if ok else "FIX  ") + name + ("" if ok else "  -> " + hint))
print("torch", torch.__version__)
try:
    import sacrebleu
except ImportError:
    import subprocess, sys; subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sacrebleu"]); import sacrebleu

## Provided: the building blocks from Lectures 32 and 33

In [ ]:
def attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None: scores = scores.masked_fill(~mask, float("-inf"))
    w = F.softmax(scores, dim=-1)
    return w @ V, w
def split_heads(x, h):
    B, n, d = x.shape
    return x.view(B, n, h, d // h).transpose(1, 2)
def merge_heads(x):
    B, h, n, dk = x.shape
    return x.transpose(1, 2).reshape(B, n, h * dk)
class MultiHeadAttention(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        assert d % h == 0
        self.d, self.h = d, h
        self.W_q = nn.Linear(d, d, bias=False); self.W_k = nn.Linear(d, d, bias=False)
        self.W_v = nn.Linear(d, d, bias=False); self.W_o = nn.Linear(d, d, bias=False)
    def forward(self, q_in, kv_in, mask=None):
        Q, K, V = split_heads(self.W_q(q_in), self.h), split_heads(self.W_k(kv_in), self.h), split_heads(self.W_v(kv_in), self.h)
        heads, w = attention(Q, K, V, mask)
        return self.W_o(merge_heads(heads)), w
class FeedForward(nn.Module):
    def __init__(self, d, d_ff):
        super().__init__(); self.l1 = nn.Linear(d, d_ff); self.l2 = nn.Linear(d_ff, d)
    def forward(self, x): return self.l2(F.relu(self.l1(x)))
class EncoderBlock(nn.Module):
    def __init__(self, d, h, d_ff, p=0.1):
        super().__init__()
        self.attn, self.ffn = MultiHeadAttention(d, h), FeedForward(d, d_ff)
        self.ln1, self.ln2, self.drop = nn.LayerNorm(d), nn.LayerNorm(d), nn.Dropout(p)
    def forward(self, x, mask=None):
        x = self.ln1(x + self.drop(self.attn(x, x, mask)[0]))
        return self.ln2(x + self.drop(self.ffn(x)))
def sinusoidal_pe(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    pos = torch.arange(max_len).unsqueeze(1).float(); div = torch.pow(10000.0, torch.arange(0, d_model, 2).float() / d_model)
    pe[:, 0::2] = torch.sin(pos / div); pe[:, 1::2] = torch.cos(pos / div); return pe
print("ok")

## Task 1: The causal mask

The decoder must not look at words it has not produced yet. Write `causal_mask(T)`: a `(T, T)` boolean matrix, `True` where position *i* may look at position *j* (that is, `j <= i`).

Then write `additive_view(mask)`: the same matrix as it appears on Page 9 of the notes, `0` where allowed and `-inf` where hidden (it is what gets added to the scores before the softmax).

In [ ]:
def causal_mask(T):
    return None        # YOUR CODE: lower-triangular boolean matrix (T, T)

def additive_view(mask):
    return None        # YOUR CODE: float matrix, 0.0 where mask is True and -inf where False

In [ ]:
m = causal_mask(4)
check("shape (4, 4) and boolean", tuple(m.shape) == (4, 4) and m.dtype == torch.bool)
check("row 0 sees only itself", m[0].tolist() == [True, False, False, False])
check("last row sees everything", bool(m[3].all()))
print(additive_view(m))
check("additive view has -inf above the diagonal", float(additive_view(m)[0, 1]) == float("-inf") and float(additive_view(m)[1, 0]) == 0.0)
w = F.softmax(torch.randn(4, 4) + additive_view(m), dim=-1)
check("after softmax the hidden weights are exactly 0", float(w.triu(1).abs().max()) == 0.0 and torch.allclose(w.sum(-1), torch.ones(4)))

## Task 2: The decoder block

Three sub-layers, each followed by dropout, a residual connection and layer norm (post-norm):

1. **Masked self-attention** on the target words so far (`tgt_mask`).
2. **Cross-attention**: queries from the decoder, keys and values from the encoder output `memory` (`src_mask` hides source padding).
3. **Feed-forward**.

Return `(output, cross_weights)`; the weights are what we plot later.

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, d, h, d_ff, p=0.1):
        super().__init__()
        self.self_attn, self.cross_attn, self.ffn = MultiHeadAttention(d, h), MultiHeadAttention(d, h), FeedForward(d, d_ff)
        self.ln1, self.ln2, self.ln3, self.drop = nn.LayerNorm(d), nn.LayerNorm(d), nn.LayerNorm(d), nn.Dropout(p)
    def forward(self, y, memory, tgt_mask, src_mask):
        # YOUR CODE 1: y = ln1( y + drop( self_attn(y, y, tgt_mask)[0] ) )
        # YOUR CODE 2: a, w = cross_attn(y, memory, src_mask)   (queries = y, keys/values = memory) ; y = ln2( y + drop(a) )
        # YOUR CODE 3: y = ln3( y + drop( ffn(y) ) ) ; return y, w
        return None, None

In [ ]:
torch.manual_seed(0)
blk = DecoderBlock(16, 4, 32); blk.eval()
y, mem = torch.randn(1, 5, 16), torch.randn(1, 7, 16)
tm = causal_mask(5)[None, None]; sm = torch.ones(1, 1, 1, 7, dtype=torch.bool)
out, w = blk(y, mem, tm, sm)
check("output shape (1, 5, 16)", tuple(out.shape) == (1, 5, 16))
check("cross weights (1, 4 heads, 5 target words, 7 source words)", tuple(w.shape) == (1, 4, 5, 7))
y2 = y.clone(); y2[0, 3:] = torch.randn(2, 16) * 5                 # change only FUTURE words (positions 3, 4)
d_future = float((blk(y2, mem, tm, sm)[0][0, :3] - out[0, :3]).abs().max())
mem2 = mem + 1.0
d_mem = float((blk(y, mem2, tm, sm)[0] - out).abs().max())
print("changing future target words moves the earlier outputs by %.1e; changing the encoder memory moves the output by %.3f" % (d_future, d_mem))
check("no leakage from the future", d_future < 1e-5, "is the causal mask applied to the SELF-attention?")
check("the decoder really uses the encoder memory", d_mem > 1e-3, "are keys/values of cross-attention taken from memory?")

## Data: the same corpus as Lectures 30 and 31
Urdu SOV sentences → English SVO sentences, 1,850 unique pairs, 85% train / 15% validation with the same shuffle seed (42). Short = 4 to 6 Urdu words, medium = 9 to 14, long = 16 to 20.

In [ ]:
import random

TRANSLIT = {"لڑکا":"larka","لڑکی":"larki","استاد":"ustaad","بچہ":"bacha","علی":"Ali","سارہ":"Sara","باپ":"baap","ماں":"maan",
 "ڈاکٹر":"doctor","نرس":"nurse","پڑھتا":"parhta","پڑھتی":"parhti","کھاتا":"khata","کھاتی":"khati","پیتا":"peeta","پیتی":"peeti",
 "دیکھتا":"dekhta","دیکھتی":"dekhti","لکھتا":"likhta","لکھتی":"likhti","ہے":"hai","اور":"aur","کتاب":"kitab","اخبار":"akhbar",
 "خط":"khat","روٹی":"roti","سیب":"saib","چاول":"chawal","چائے":"chai","پانی":"pani","دودھ":"doodh","فلم":"film","گاڑی":"gaari",
 "پرندہ":"parinda","کہانی":"kahani","نام":"naam","گھر":"ghar","سکول":"school","میں":"mein","<EOS>":"<EOS>"}
def build_pairs(n_two=700, n_three=700, seed=0):
    r = random.Random(seed)
    subj = [("لڑکا","the boy","m"),("لڑکی","the girl","f"),("استاد","the teacher","m"),("بچہ","the child","m"),
            ("علی","Ali","m"),("سارہ","Sara","f"),("باپ","the father","m"),("ماں","the mother","f"),
            ("ڈاکٹر","the doctor","m"),("نرس","the nurse","f")]
    verbs = {  # (urdu masc, urdu fem, english 3rd person)
      "read":("پڑھتا","پڑھتی","reads",[("کتاب","the book"),("اخبار","the newspaper"),("خط","the letter")]),
      "eat": ("کھاتا","کھاتی","eats",[("روٹی","bread"),("سیب","the apple"),("چاول","rice")]),
      "drink":("پیتا","پیتی","drinks",[("چائے","tea"),("پانی","water"),("دودھ","milk")]),
      "see": ("دیکھتا","دیکھتی","sees",[("فلم","the movie"),("گاڑی","the car"),("پرندہ","the bird")]),
      "write":("لکھتا","لکھتی","writes",[("کہانی","the story"),("نام","the name"),("خط","the letter")]),
    }
    places = [("", ""), ("گھر میں","at home"), ("سکول میں","at school")]
    simple = []
    for su,se,g in subj:
        for vk,(vm,vf,ve,objs) in verbs.items():
            for ou,oe in objs:
                for pu,pe in places:
                    u = f"{su} {pu} {ou} {vm if g=='m' else vf} ہے".replace("  "," ")
                    e = f"{se[0].upper()+se[1:]} {ve} {oe} {pe}".strip()
                    simple.append((u,e))
    simple = sorted(set(simple))
    def join(parts):
        u = " اور ".join(p[0] for p in parts)
        es = [parts[0][1]] + [(p[1][0].lower() + p[1][1:]) if p[1].startswith("The ") else p[1] for p in parts[1:]]
        return (u, " and ".join(es))
    comp = set()
    while len(comp) < n_two:
        comp.add(join(r.sample(simple, 2)))
    while len(comp) < n_two + n_three:
        comp.add(join(r.sample(simple, 3)))
    pairs = simple + sorted(comp)
    r.shuffle(pairs)
    return pairs

pairs = build_pairs(); random.Random(42).shuffle(pairs)
split = int(0.85 * len(pairs)); train, valid = pairs[:split], pairs[split:]
print(len(train), "train /", len(valid), "validation pairs")
for u, e in valid[:2]: print(" ", " ".join(TRANSLIT[w] for w in u.split()), "->", e)
def make_vocab(sents):
    v = ["<pad>", "<start>", "<end>"] + sorted({w for s in sents for w in s.split()}); return {w: i for i, w in enumerate(v)}
src_v, tgt_v = make_vocab([a for a, b in pairs]), make_vocab([b for a, b in pairs]); inv = {i: w for w, i in tgt_v.items()}
PAD, BOS, EOS = 0, 1, 2
def pad_batch(seqs):
    L_ = max(map(len, seqs)); x = torch.zeros(len(seqs), L_, dtype=torch.long)
    for i, s in enumerate(seqs): x[i, :len(s)] = torch.tensor(s)
    return x
def encode_pairs(ps):
    src = pad_batch([[src_v[w] for w in a.split()] for a, b in ps])
    tgt_in = pad_batch([[BOS] + [tgt_v[w] for w in b.split()] for a, b in ps])
    tgt_out = pad_batch([[tgt_v[w] for w in b.split()] + [EOS] for a, b in ps])
    return src, tgt_in, tgt_out
def bucket(ps, lo, hi): return [p for p in ps if lo <= len(p[0].split()) <= hi]
print("vocab sizes (src/tgt):", len(src_v), len(tgt_v))

## Task 3: Teacher-forcing inputs

For a target sentence with word IDs `ids`, the decoder INPUT is `[BOS] + ids` and the decoder TARGET is `ids + [EOS]`: the input is the target shifted right by one (Page 11). Write `shift_target(ids)` returning `(in_ids, out_ids)`.

In [ ]:
def shift_target(ids):
    return None, None        # YOUR CODE

In [ ]:
a, b = shift_target([5, 6, 7])
check("input", a == [1, 5, 6, 7]); check("target", b == [5, 6, 7, 2])
check("same length", len(a) == len(b))
s_, ti_, to_ = encode_pairs(train[:1])
check("matches the provided encoder", ti_[0, 0].item() == BOS and to_[0, ti_.shape[1] - 1].item() == EOS)
print("example: input", [inv.get(i, '<start>') for i in ti_[0].tolist()], "\n         target", [inv.get(i, '<end>') for i in to_[0].tolist()])

## Provided: the full Transformer and its training loop
Embedding × √d + positions, N encoder blocks, N decoder blocks, a linear layer to the vocabulary. Training: cross-entropy ignoring padding, Adam (β₂ = 0.98), a learning rate that **warms up** for 200 steps and then follows a **cosine decay** (Page 11).

In [ ]:
class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d=64, h=4, dff=128, n_layers=2, p=0.1, max_len=64, use_pe=True):
        super().__init__()
        self.d, self.use_pe = d, use_pe
        self.src_emb, self.tgt_emb = nn.Embedding(src_vocab, d, padding_idx=0), nn.Embedding(tgt_vocab, d, padding_idx=0)
        self.register_buffer("pe", sinusoidal_pe(max_len, d)); self.drop = nn.Dropout(p)
        self.enc = nn.ModuleList([EncoderBlock(d, h, dff, p) for _ in range(n_layers)])
        self.dec = nn.ModuleList([DecoderBlock(d, h, dff, p) for _ in range(n_layers)])
        self.out = nn.Linear(d, tgt_vocab)
    def embed(self, ids, emb):
        x = emb(ids) * math.sqrt(self.d)
        if self.use_pe: x = x + self.pe[: ids.shape[1]]
        return self.drop(x)
    def encode(self, src):
        src_mask = (src != PAD)[:, None, None, :]                       # (B, 1, 1, S): True = real word
        x = self.embed(src, self.src_emb)
        for blk in self.enc: x = blk(x, src_mask)
        return x, src_mask
    def decode(self, tgt_in, memory, src_mask):
        T = tgt_in.shape[1]
        tgt_mask = causal_mask(T)[None, None] & (tgt_in != PAD)[:, None, None, :]     # causal AND not padding
        y = self.embed(tgt_in, self.tgt_emb); ws = []
        for blk in self.dec: y, w = blk(y, memory, tgt_mask, src_mask); ws.append(w)
        return self.out(y), ws
    def forward(self, src, tgt_in):
        memory, src_mask = self.encode(src)
        return self.decode(tgt_in, memory, src_mask)[0]
    @torch.no_grad()
    def greedy(self, src, max_len=30):
        self.eval(); memory, src_mask = self.encode(src)
        ys = torch.full((src.shape[0], 1), BOS, dtype=torch.long)
        for _ in range(max_len):
            logits = self.decode(ys, memory, src_mask)[0][:, -1]
            ys = torch.cat([ys, logits.argmax(-1, keepdim=True)], 1)
            if bool((ys == EOS).any(1).all()): break
        return ys[:, 1:]

CFG = {'d': 64, 'h': 4, 'dff': 128, 'L': 2, 'p': 0.1, 'ep': 40, 'lr': 0.001, 'warm': 200, 'bs': 32}
def train_transformer(train_pairs, seed, use_pe=True, **over):
    c = dict(CFG); c.update(over); torch.manual_seed(seed); random.seed(seed)
    model = Transformer(len(src_v), len(tgt_v), c["d"], c["h"], c["dff"], c["L"], c["p"], use_pe=use_pe)
    opt = torch.optim.Adam(model.parameters(), c["lr"], betas=(0.9, 0.98), eps=1e-9)
    S, TI, TO = encode_pairs(train_pairs); n = len(train_pairs); steps_per = math.ceil(n / c["bs"]); total = c["ep"] * steps_per; step = 0
    for ep in range(c["ep"]):
        model.train(); perm = torch.randperm(n)
        for i in range(0, n, c["bs"]):
            b = perm[i:i + c["bs"]]; step += 1
            lr = c["lr"] * step / c["warm"] if step <= c["warm"] else c["lr"] * 0.5 * (1 + math.cos(math.pi * min(1, (step - c["warm"]) / max(1, total - c["warm"]))))
            for g in opt.param_groups: g["lr"] = lr
            logits = model(S[b], TI[b])
            loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), TO[b].reshape(-1), ignore_index=PAD)
            opt.zero_grad(); loss.backward(); opt.step()
    return model

def translate_all(model, ps):
    S = encode_pairs(ps)[0]; hyps = []
    for i in range(0, len(ps), 128):
        for row in model.greedy(S[i:i + 128]).tolist():
            ws = []
            for t in row:
                if t == EOS: break
                ws.append(inv[t])
            hyps.append(" ".join(ws))
    return hyps
def score(model, ps):
    hyps = translate_all(model, ps); refs = [b for a, b in ps]
    return sacrebleu.corpus_bleu(hyps, [refs]).score, sum(h == r for h, r in zip(hyps, refs)) / len(ps)
BUCKETS = {"short": (4, 6), "medium": (9, 14), "long": (16, 20)}
def score_buckets(model): return {k: score(model, bucket(valid, lo, hi))[1] for k, (lo, hi) in BUCKETS.items()}
n_params = lambda m: sum(p.numel() for p in m.parameters())
print("config", CFG)

### Train the baseline (one seed, about 1 minute)

In [ ]:
t0 = time.time(); base = train_transformer(train, 0)
bleu0, em0 = score(base, valid)
print("seed 0: BLEU %.1f  exact match %.3f  parameters %d  (%.0f s)" % (bleu0, em0, n_params(base), time.time() - t0))
for u, e in valid[:3]:
    hyp = translate_all(base, [(u, e)])[0]
    print(" ".join(TRANSLIT[w] for w in u.split()), "\n   ref:", e, "\n   hyp:", hyp, "   OK" if hyp == e else "   WRONG")

## Task 4: Greedy decoding, written by you

The provided `model.greedy` already exists. Write your own `my_greedy(model, src, max_len=30)` for a batch of source sentences: encode once, start from `<start>`, and repeat: run the decoder on the words so far, take the **argmax at the LAST position**, append it. Stop when every sentence has produced `<end>`. Return the generated IDs without the `<start>` column.

In [ ]:
@torch.no_grad()
def my_greedy(model, src, max_len=30):
    model.eval()
    memory, src_mask = model.encode(src)
    ys = torch.full((src.shape[0], 1), BOS, dtype=torch.long)
    for _ in range(max_len):
        # YOUR CODE: logits for the last position, next = argmax, append to ys, stop when every row has EOS
        pass
    return ys[:, 1:]

In [ ]:
S50 = encode_pairs(valid[:50])[0]
mine, ref = my_greedy(base, S50), base.greedy(S50)
check("same shape as the provided greedy", mine.shape == ref.shape)
check("identical translations on 50 validation sentences", torch.equal(mine, ref))

## Task 5: Teacher forcing hides mistakes

Measure two numbers on the validation set for the baseline model:

* `tf_token_acc`: with TEACHER FORCING (feed the true previous words), the fraction of target positions (excluding padding) where the argmax equals the true next word.
* `free_exact`: with FREE running (greedy decoding), the fraction of sentences translated exactly (that is `em0` above).

Why can the first be so much higher than the second?

In [ ]:
S, TI, TO = encode_pairs(valid)
base.eval()
with torch.no_grad():
    pred = None            # YOUR CODE: argmax over the vocabulary of base(S, TI), shape (B, T)
mask = TO != PAD
tf_token_acc = None        # YOUR CODE: fraction of non-pad positions where pred == TO
free_exact = em0

In [ ]:
print("teacher-forced token accuracy %.3f   free-running exact match %.3f" % (tf_token_acc, free_exact))
check("token accuracy is well above exact match", tf_token_acc > free_exact + 0.05)
avg_len = float(mask.sum(1).float().mean()); print("average target length: %.1f tokens, so one early mistake spoils the whole sentence" % avg_len)

## Experiment 1: three seeds, with the length buckets
Then we compare with the numbers we measured earlier on the SAME validation set. **Care:** the Lecture 31 models were trained with different code, 12 epochs, hidden size 256 and Adam 0.001; our Transformer is deliberately small. This is a comparison of two reasonable setups, not of two architectures in the abstract.

In [ ]:
E1 = []
for s in (0, 1, 2):
    m = base if s == 0 else train_transformer(train, s)
    b, e = score(m, valid); bk = score_buckets(m)
    E1.append({"seed": s, "bleu": b, "em": e, **bk, "params": n_params(m)})
    print(f"seed {s}: BLEU {b:.1f}  exact {e:.3f}  short {bk['short']:.3f}  medium {bk['medium']:.3f}  long {bk['long']:.3f}")
R["e1"] = E1
L31 = {"LSTM, no attention": {"bleu": 79.1, "em": 0.475, "long": 0.04}, "LSTM + Bahdanau attention": {"bleu": 100.0, "em": 1.0, "long": 1.0}}   # measured in Lecture 31 (seed 42)
R["l31"] = L31; print("Lecture 31 (seed 42, hidden 256, 12 epochs):", L31)

## Experiment 2: take parts away
Two seeds each (0 and 1), same data, same settings, one change at a time:

* **no positional encoding**
* **1 head** (d = 64, so the single head has 64 columns) and **8 heads** (8 columns each) instead of 4

In [ ]:
E2 = {}
for name, kw in {"no positions": {"use_pe": False}, "1 head": {"h": 1}, "8 heads": {"h": 8}}.items():
    runs = []
    for s in (0, 1):
        use_pe = kw.get("use_pe", True); over = {k: v for k, v in kw.items() if k != "use_pe"}
        m = train_transformer(train, s, use_pe=use_pe, **over)
        b, e = score(m, valid); runs.append({"bleu": b, "em": e})
    E2[name] = runs
    print(f"{name:13s} BLEU {[round(r['bleu'], 1) for r in runs]}  exact {[round(r['em'], 3) for r in runs]}")
E2["baseline (4 heads)"] = [{"bleu": r["bleu"], "em": r["em"]} for r in E1[:2]]
R["e2"] = E2

## Experiment 3: does more data help?
Our training set has only about 1,570 pairs. We build a larger set of the same kind (about 8,450 unique pairs), remove every pair whose Urdu sentence is in the validation set, and train on 3,000 and on 6,000 pairs (one seed each, same number of epochs). Validation stays the same.

In [ ]:
big = build_pairs(4000, 4000, 0)
valid_src = {a for a, b in valid}
big = [p for p in big if p[0] not in valid_src]
random.Random(7).shuffle(big)
print("larger pool:", len(big), "pairs (validation sentences removed)")
E3 = [{"n_train": len(train), "bleu": E1[0]["bleu"], "em": E1[0]["em"], "long": E1[0]["long"]}]
for n in (3000, 6000):
    t0 = time.time(); m = train_transformer(big[:n], 0)
    b, e = score(m, valid); bk = score_buckets(m)
    E3.append({"n_train": n, "bleu": b, "em": e, "long": bk["long"]})
    print(f"train on {n}: BLEU {b:.1f}  exact {e:.3f}  long {bk['long']:.3f}  ({time.time() - t0:.0f} s)")
R["e3"] = E3

### Task 6: Read the three experiments

Set the variables from the numbers printed above (means over the seeds that were run).

In [ ]:
mean = lambda a: sum(a) / len(a)
em_base = None            # mean exact match of the 3 baseline seeds (E1)
no_pos_hurts = None       # True if the mean exact match WITHOUT positions is below the 4-head baseline mean (E2)
more_data_helps = None    # True if exact match with 6000 training pairs is above the 1,572-pair run (E3)

In [ ]:
check("em_base", abs(em_base - sum(r["em"] for r in E1) / 3) < 1e-9)
check("no_pos_hurts", no_pos_hurts == (sum(r["em"] for r in E2["no positions"]) / 2 < sum(r["em"] for r in E2["baseline (4 heads)"]) / 2))
check("more_data_helps", more_data_helps == (E3[-1]["em"] > E3[0]["em"]))
print("em_base %.3f, no_pos_hurts %s, more_data_helps %s" % (em_base, no_pos_hurts, more_data_helps))

## Where does the decoder look? Cross-attention

In [ ]:
u, e = valid[1]
S, TI, TO = encode_pairs([(u, e)]); base.eval()
with torch.no_grad():
    memory, sm = base.encode(S); logits, ws = base.decode(TI, memory, sm)
w = ws[-1][0].mean(0).numpy()                     # last decoder block, averaged over its 4 heads: (target words, source words)
src_w = [TRANSLIT[x] for x in u.split()]; tgt_w = [inv[i] for i in TO[0].tolist()[:-1]] + ["<end>"]
fig, ax = plt.subplots(figsize=(max(6, 0.55 * len(src_w) + 2), max(3, 0.38 * len(tgt_w) + 1.4)))
im = ax.imshow(w, cmap="Blues", vmin=0, vmax=max(0.5, w.max()))
ax.set_xticks(range(len(src_w))); ax.set_xticklabels(src_w, rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(len(tgt_w))); ax.set_yticklabels(tgt_w, fontsize=8); ax.set_xlabel("source words (Urdu, transliterated)"); ax.set_ylabel("generated word")
ax.set_title("Cross-attention, last decoder block, average of 4 heads", fontsize=9); fig.tight_layout(); fig.savefig("fig_cross.png", dpi=170); plt.close(fig)
print("saved fig_cross.png   (sentence:", " ".join(src_w), "->", e, ")")
srcmax = [src_w[int(i)] for i in w.argmax(1)]
print("most-attended source word for each generated word:", list(zip(tgt_w, srcmax)))
R["cross_demo"] = {"src": src_w, "tgt": tgt_w, "argmax_src": srcmax}

## Wrap-up
- Decoder block = masked self-attention + cross-attention + feed-forward, each with add & norm.
- The causal mask lets us train all positions at once; generation is still one word at a time.
- Teacher forcing makes per-word accuracy look much better than whole-sentence accuracy; errors accumulate at inference time.
- A Transformer is not automatically better: on our tiny corpus it needed care and more data (Experiment 3). Next lecture: BERT, GPT, T5, pre-training and why attention gets expensive for long inputs.

**Think about it:** the cross-attention picture is a clue, not a proof of how the model translates. Why can't we conclude "the model aligns words like a dictionary" from one heatmap?

In [ ]:
print("RESULT_JSON " + json.dumps(R))